# Post-processing: attach residual force columns

Central place to append an `f_residual` column to log data. Given one or more data
folders (under `data/training`), every `.csv` in them gets a residual column computed
by a `BemtModel` against a fitted lookup table. The augmented data is then used for
network training.

The model is only a vehicle for the correct blade and drone geometry, so the same
workflow serves both the full-vehicle and single-propeller cases -- just point the
config cell at the right blade, drone, lookup table and folders.

In [ ]:
import sys
import os
import numpy as np
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Add the 2nd level parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(parent_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import data_factory
from inflow_model.blade_params import P600_Blade, APC_8x6
from bemt_model import BemtModel
from fitting_config import FittingConfig
from post_processing import make_residual_force_columns
from drone import parameters
import inflow_model.propeller_lookup_table as propeller_lookup_table

Configure the blade, drone, fitted lookup table and the data folders to process.

In [ ]:
# Folders under data/training whose CSVs get an f_residual column. Add as many as needed
# (e.g. both a training and a validation folder).
data_subfolders = [
    "wind_free_space_training_cfd_our_proposal",
]

blade = P600_Blade()
drone = parameters.P600()
lookup_table_name = "p600_single_rotor_after_bemt_fit"

# BemtModel needs a ModelConfig; the full-vehicle config's resolution settings are fine here.
model_config = FittingConfig.from_yaml("config_full_vehicle.yaml").model
model = BemtModel(blade, drone, model_config)
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader(lookup_table_name)

Attach the residual force column to every CSV in each folder. Re-run after rebuilding
the lookup table to refresh the columns.

In [ ]:
factory = data_factory.FittingFactory()
for subfolder in data_subfolders:
    data_list = data_factory.generate_data_list(subfolder, '.csv')
    datasets = factory.prepare_datasets(data_list)
    for dataset in datasets:
        print(f"Filling residual force column of {os.path.relpath(dataset.path_to_data_file)}")
        make_residual_force_columns(model, dataset, lookup_table)